# Corte 3, Punto 1: deduplicación y capas silver y gold

Por: Gabriel Rodriguez y Jonatan Palomares

**Subtareas:** T1.1 a T1.6 (diagnóstico de duplicados, silver, gold, tablas resumen, reconciliación y linaje).

Entrada: bronze. Salidas: `silver/`, `gold/` y `procesado/`.

_Por hacer._

## Configuración

Las rutas están en `src/pamd/config.py` y la sesión de Spark con MinIO en `src/pamd/spark_session.py`. Las credenciales vienen de las variables de entorno definidas en `.env`.

In [1]:
# Importación de librerías

import os
from time import perf_counter
from statistics import median
import random
import math
import json
import urllib.request
from IPython.display import display, Markdown

import matplotlib.pyplot as plt

from pyspark.sql import SparkSession, Window, functions as F, types as T
from pyspark.sql.types import DateType, TimestampType
from pyspark import StorageLevel

from pamd.config import (
    INPUT_CSV,
    BRONZE_PARQUET_UNPARTITIONED,
    BRONZE_PARQUET,
    QUARANTINE_PARQUET,
    SILVER_PARQUET,
    SILVER_DESCARTADOS_PARQUET,
    GOLD_PARQUET,
    GOLD_EXCLUIDOS_PARQUET,
    WRITE_MODE,
    OUTPUT_PARTITIONS,
)
from pamd.spark_session import get_spark

# Spark conectado a MinIO; las credenciales se leen del archivo .env
spark = get_spark()


## T1.1 Diagnóstico de duplicados y elección de la clave natural

El bronze conserva todas las filas válidas del CSV, pero un mismo proceso de contratación puede aparecer varias veces (nuevas publicaciones, lotes o varias adjudicaciones). Antes de deduplicar hay que medir cuánto se repite cada clave candidata y entender por qué, para elegir la clave que define **un contrato adjudicado**.

Claves candidatas:
- **Fila completa:** filas idénticas en todas las columnas (duplicado exacto).
- **`id_del_proceso`:** un registro por proceso.
- **`id_del_proceso` + `id_adjudicacion`:** un registro por adjudicación dentro de un proceso.
- **`id_adjudicacion`:** identificador de la adjudicación por sí solo.

Se mide sobre todo el bronze y sobre los registros con `adjudicado = True`, que son los que pasarán a gold.

In [2]:
# T1.1 · Bronze de entrada
bronze = spark.read.parquet(BRONZE_PARQUET_UNPARTITIONED)

# fecha_ingesta_utc la pone el proceso de carga; no describe el contrato
columnas_contenido = [c for c in bronze.columns if c != "fecha_ingesta_utc"]

filas_bronze = bronze.count()
filas_adjudicadas = bronze.filter(F.col("adjudicado") == True).count()
print(f"Filas en el bronze:      {filas_bronze:,}")
print(f"Filas con adjudicado=True: {filas_adjudicadas:,}")
print(f"Columnas de contenido:   {len(columnas_contenido)}")

Filas en el bronze:      9,099,432
Filas con adjudicado=True: 872,756
Columnas de contenido:   59


### Tabla de diagnóstico por clave candidata

*Filas sobrantes* = filas − claves distintas: son las filas que desaparecerían si se dejara una sola por clave.

In [3]:
# T1.1 · Tabla de diagnóstico: filas, claves distintas y filas sobrantes por clave candidata
from functools import reduce
from operator import or_
import pandas as pd

claves_candidatas = {
    "fila completa (duplicado exacto)": columnas_contenido,
    "id_del_proceso": ["id_del_proceso"],
    "id_del_proceso + id_adjudicacion": ["id_del_proceso", "id_adjudicacion"],
    "id_adjudicacion": ["id_adjudicacion"],
}

def diagnosticar(df, universo):
    filas = df.count()
    resultado = []
    for nombre, columnas in claves_candidatas.items():
        if len(columnas) > 2:
            # Fila completa: se compara un hash de 64 bits de todas las columnas
            # (con 9 millones de filas la probabilidad de colisión es despreciable)
            distintas = df.select(F.xxhash64(*columnas).alias("h")).distinct().count()
            nulas = "n/a"
        else:
            distintas = df.select(*columnas).distinct().count()
            nulas = df.filter(reduce(or_, [F.col(c).isNull() for c in columnas])).count()
        resultado.append({
            "universo": universo,
            "clave candidata": nombre,
            "filas": filas,
            "claves distintas": distintas,
            "filas sobrantes": filas - distintas,
            "% sobrante": round(100 * (filas - distintas) / filas, 2) if filas else 0.0,
            "filas con clave nula": nulas,
        })
    return resultado

diagnostico_claves = pd.DataFrame(
    diagnosticar(bronze, "todo el bronze")
    + diagnosticar(bronze.filter(F.col("adjudicado") == True), "solo adjudicados")
)
display(diagnostico_claves)

,universo,clave candidata,filas,claves distintas,filas sobrantes,% sobrante,filas con clave nula
0,todo el bronze,fila completa (duplicado exacto),9099432,8963967,135465,1.49,n/a
1,todo el bronze,id_del_proceso,9099432,8667332,432100,4.75,0
2,todo el bronze,id_del_proceso + id_adjudicacion,9099432,8694126,405306,4.45,8226676
3,todo el bronze,id_adjudicacion,9099432,547419,8552013,93.98,8226676
4,solo adjudicados,fila completa (duplicado exacto),872756,817259,55497,6.36,n/a
5,solo adjudicados,id_del_proceso,872756,520624,352132,40.35,0
6,solo adjudicados,id_del_proceso + id_adjudicacion,872756,547418,325338,37.28,0
7,solo adjudicados,id_adjudicacion,872756,547418,325338,37.28,0


### Distribución de repeticiones por proceso

In [4]:
# T1.1 · ¿Cuántas veces se repite cada id_del_proceso?
repeticiones_proceso = (
    bronze.groupBy("id_del_proceso").count()
    .withColumn("veces", F.least(F.col("count"), F.lit(10)))
    .groupBy("veces")
    .agg(F.count("*").alias("procesos"), F.sum("count").alias("filas"))
    .orderBy("veces")
    .collect()
)
repeticiones_proceso = pd.DataFrame([r.asDict() for r in repeticiones_proceso])
repeticiones_proceso["veces"] = repeticiones_proceso["veces"].map(lambda v: "10 o más" if v == 10 else str(v))
display(repeticiones_proceso)

,veces,procesos,filas
0,1,8583881,8583881
1,2,50766,101532
2,3,10958,32874
3,4,8696,34784
4,5,2734,13670
5,6,2271,13626
6,7,761,5327
7,8,1041,8328
8,9,1310,11790
9,10 o más,4914,293620


### Qué cambia entre las filas de un mismo proceso

Si en un proceso repetido cambia `id_adjudicacion`, las filas son adjudicaciones distintas (lotes o varios contratistas). Si solo cambian fechas de publicación o el estado, son versiones sucesivas del mismo proceso.

In [5]:
# T1.1 · Dentro de un mismo id_del_proceso repetido, ¿qué columnas cambian?
# Para cada proceso repetido se cuenta cuántos valores distintos tiene cada columna
# (se usa un hash por valor para no cargar textos largos en memoria; el nulo cuenta como valor).
procesos_repetidos = (
    bronze.groupBy("id_del_proceso").count()
    .filter(F.col("count") > 1)
    .select("id_del_proceso")
)
columnas_a_revisar = [c for c in columnas_contenido if c != "id_del_proceso"]

variacion = (
    bronze.join(procesos_repetidos, "id_del_proceso", "left_semi")
    .groupBy("id_del_proceso")
    .agg(*[F.size(F.collect_set(F.xxhash64(F.col(c)))).alias(c) for c in columnas_a_revisar])
)
total_repetidos = procesos_repetidos.count()

fila = variacion.agg(*[F.sum(F.when(F.col(c) > 1, 1).otherwise(0)).alias(c) for c in columnas_a_revisar]).first()
columnas_que_cambian = pd.DataFrame(
    [(c, fila[c] or 0, round(100 * (fila[c] or 0) / total_repetidos, 2) if total_repetidos else 0.0)
     for c in columnas_a_revisar],
    columns=["columna", "procesos repetidos donde cambia", "% de procesos repetidos"],
).sort_values("procesos repetidos donde cambia", ascending=False)

print(f"Procesos con más de una fila: {total_repetidos:,}")
display(columnas_que_cambian[columnas_que_cambian["procesos repetidos donde cambia"] > 0].reset_index(drop=True))

Procesos con más de una fila: 83,451


,columna,procesos repetidos donde cambia,% de procesos repetidos
0,valor_total_adjudicacion,22389,26.83
1,id_adjudicacion,18308,21.94
2,nombre_proveedor_adjudicado,14393,17.25
3,codigo_proveedor,14393,17.25
4,fecha_adjudicacion,14099,16.89
5,nit_proveedor_adjudicado,10433,12.50
6,departamento_proveedor,9889,11.85
7,ciudad_proveedor,9541,11.43
8,nombre_adjudicador,2582,3.09


### Muestra de procesos repetidos para revisión manual

In [6]:
# T1.1 · Muestra reproducible de procesos repetidos para revisarlos a mano
# Se eligen los 5 procesos repetidos con el hash más bajo: siempre salen los mismos.
columnas_muestra = [
    "id_del_proceso", "referencia_del_proceso", "fase", "estado_procedimiento",
    "adjudicado", "id_adjudicacion", "numero_lotes",
    "fecha_publicacion_proceso", "fecha_ultima_publicacion", "fecha_adjudicacion",
    "precio_base", "valor_total_adjudicacion",
    "nit_proveedor_adjudicado", "nombre_proveedor_adjudicado",
]
ids_muestra = [
    r.id_del_proceso
    for r in procesos_repetidos.orderBy(F.xxhash64("id_del_proceso"), "id_del_proceso").limit(5).collect()
]
muestra_repetidos = (
    bronze.filter(F.col("id_del_proceso").isin(ids_muestra))
    .select(*columnas_muestra)
    .orderBy("id_del_proceso", "fecha_ultima_publicacion", "id_adjudicacion")
    .collect()
)
muestra_repetidos = pd.DataFrame([r.asDict() for r in muestra_repetidos], columns=columnas_muestra)
pd.set_option("display.max_columns", None)
display(muestra_repetidos)

,id_del_proceso,referencia_del_proceso,fase,estado_procedimiento,adjudicado,id_adjudicacion,numero_lotes,fecha_publicacion_proceso,fecha_ultima_publicacion,fecha_adjudicacion,precio_base,valor_total_adjudicacion,nit_proveedor_adjudicado,nombre_proveedor_adjudicado
0,CO1.REQ.407030,PN DIBIE SA 023 2018 (Presentación de oferta),Presentación de oferta,Seleccionado,True,CO1.AWD.300105,0,2018-04-10,2018-04-10,NaT,103805319.00,101533300.00,None,VICARTECHZ SAS
1,CO1.REQ.407030,PN DIBIE SA 023 2018 (Presentación de oferta),Presentación de oferta,Seleccionado,True,CO1.AWD.301134,0,2018-04-10,2018-04-10,2018-06-08,103805319.00,95532681.00,None,TEAM CORP SAS
2,CO1.REQ.5912382,CTO 091 MT 2024,Presentación de oferta,Seleccionado,False,None,0,2024-03-08,2024-03-08,NaT,50400000.00,0.00,None,None
3,CO1.REQ.5912382,CTO 091 MT 2024,Presentación de oferta,Seleccionado,False,None,0,2024-03-08,2024-03-08,NaT,50400000.00,0.00,None,None
4,CO1.REQ.6002798,4148.010.32.1.0392-2024,Presentación de oferta,Seleccionado,True,CO1.AWD.1897759,0,2024-03-23,2024-03-23,2024-04-11,1016904076.00,1335411458.00,900528294,Fundación Bibliotec
5,CO1.REQ.6002798,4148.010.32.1.0392-2024,Presentación de oferta,Seleccionado,True,CO1.AWD.1897759,0,2024-03-23,2024-03-23,2024-04-11,1016904076.00,1335411458.00,900528294,Fundación Bibliotec
6,CO1.REQ.6002798,4148.010.32.1.0392-2024,Presentación de oferta,Seleccionado,True,CO1.AWD.1897759,0,2024-03-23,2024-03-23,2024-04-11,1016904076.00,1335411458.00,900528294,Fundación Bibliotec
7,CO1.REQ.6002798,4148.010.32.1.0392-2024,Presentación de oferta,Seleccionado,True,CO1.AWD.1897759,0,2024-03-23,2024-03-23,2024-04-11,1016904076.00,1335411458.00,900528294,Fundación Bibliotec
8,CO1.REQ.6002798,4148.010.32.1.0392-2024,Presentación de oferta,Seleccionado,True,CO1.AWD.1897759,0,2024-03-23,2024-03-23,2024-04-11,1016904076.00,1335411458.00,900528294,Fundación Bibliotec
9,CO1.REQ.6296380,413-MC-026-2024,Presentación de oferta,Seleccionado,True,CO1.AWD.1941961,0,2024-05-27,2024-05-27,NaT,5130000.00,4288811.00,None,SERHE H2O


In [7]:
# T1.1 · Dentro de una misma adjudicación, ¿qué cambia cuando las filas NO son duplicados exactos?
adjudicados_sin_exactos = (
    bronze.filter(F.col("adjudicado") == True)
    .dropDuplicates(columnas_contenido)
)
adjudicaciones_repetidas = (
    adjudicados_sin_exactos.groupBy("id_adjudicacion").count()
    .filter(F.col("count") > 1)
    .select("id_adjudicacion")
)
filas_adj_sin_exactos = adjudicados_sin_exactos.count()
total_adj_repetidas = adjudicaciones_repetidas.count()

columnas_adj = [c for c in columnas_contenido if c != "id_adjudicacion"]
fila_adj = (
    adjudicados_sin_exactos.join(adjudicaciones_repetidas, "id_adjudicacion", "left_semi")
    .groupBy("id_adjudicacion")
    .agg(*[F.size(F.collect_set(F.xxhash64(F.col(c)))).alias(c) for c in columnas_adj])
    .agg(*[F.sum(F.when(F.col(c) > 1, 1).otherwise(0)).alias(c) for c in columnas_adj])
    .first()
)
cambios_adjudicacion = pd.DataFrame(
    [(c, fila_adj[c] or 0, round(100 * (fila_adj[c] or 0) / total_adj_repetidas, 2) if total_adj_repetidas else 0.0)
     for c in columnas_adj],
    columns=["columna", "adjudicaciones donde cambia", "% de adjudicaciones repetidas"],
).sort_values("adjudicaciones donde cambia", ascending=False)

print(f"Filas adjudicadas sin duplicados exactos: {filas_adj_sin_exactos:,}")
print(f"Adjudicaciones distintas:                 {adjudicados_sin_exactos.select('id_adjudicacion').distinct().count():,}")
print(f"Adjudicaciones con más de una versión:    {total_adj_repetidas:,}")
display(cambios_adjudicacion[cambios_adjudicacion["adjudicaciones donde cambia"] > 0].reset_index(drop=True))

Filas adjudicadas sin duplicados exactos: 817,259
Adjudicaciones distintas:                 547,418
Adjudicaciones con más de una versión:    16,616


,columna,adjudicaciones donde cambia,% de adjudicaciones repetidas
0,valor_total_adjudicacion,16110,96.95
1,codigo_proveedor,10571,63.62
2,nombre_proveedor_adjudicado,10571,63.62
3,nit_proveedor_adjudicado,7893,47.50
4,departamento_proveedor,7333,44.13
5,ciudad_proveedor,7094,42.69
6,fecha_adjudicacion,163,0.98


In [8]:
# T1.1 · Muestra reproducible de adjudicaciones con varias versiones distintas
columnas_que_varian = cambios_adjudicacion.loc[
    cambios_adjudicacion["adjudicaciones donde cambia"] > 0, "columna"
].tolist()[:10]
ids_adj_muestra = [
    r.id_adjudicacion
    for r in adjudicaciones_repetidas.orderBy(F.xxhash64("id_adjudicacion"), "id_adjudicacion").limit(5).collect()
]
muestra_adjudicaciones = (
    adjudicados_sin_exactos.filter(F.col("id_adjudicacion").isin(ids_adj_muestra))
    .select("id_adjudicacion", "id_del_proceso", *columnas_que_varian)
    .orderBy("id_adjudicacion", *columnas_que_varian)
    .collect()
)
display(pd.DataFrame([r.asDict() for r in muestra_adjudicaciones],
                     columns=["id_adjudicacion", "id_del_proceso", *columnas_que_varian]))

,id_adjudicacion,id_del_proceso,valor_total_adjudicacion,codigo_proveedor,nombre_proveedor_adjudicado,nit_proveedor_adjudicado,departamento_proveedor,ciudad_proveedor,fecha_adjudicacion
0,CO1.AWD.1346458,CO1.REQ.3045146,50000000.00,703956128,ESTUDIOS OFTALMOLOGICOS S.A.S.,None,None,None,2022-06-22
1,CO1.AWD.1346458,CO1.REQ.3045146,55000000.00,703956128,ESTUDIOS OFTALMOLOGICOS S.A.S.,None,None,None,2022-06-22
2,CO1.AWD.1346458,CO1.REQ.3045146,75000000.00,703956128,ESTUDIOS OFTALMOLOGICOS S.A.S.,None,None,None,2022-06-22
3,CO1.AWD.1492664,CO1.REQ.3662832,106200000.00,705772788,JEILER ANDRES GALVEZ LONDOÑO,None,None,None,2022-12-13
4,CO1.AWD.1492664,CO1.REQ.3662832,213800000.00,705772788,JEILER ANDRES GALVEZ LONDOÑO,None,None,None,2022-12-13
5,CO1.AWD.1574325,CO1.REQ.4257936,90000000.00,701563397,JAVIER OSVALDO RAMIREZ MARTINEZ,None,None,None,2023-03-29
6,CO1.AWD.1574325,CO1.REQ.4257936,90000000.00,703517524,M3 SOLUCIONES INTEGRALES SAS,900413586,Atlántico,Barranquilla,2023-03-29
7,CO1.AWD.1574325,CO1.REQ.4257936,240000000.00,701563397,JAVIER OSVALDO RAMIREZ MARTINEZ,None,None,None,2023-03-29
8,CO1.AWD.1574325,CO1.REQ.4257936,240000000.00,703517524,M3 SOLUCIONES INTEGRALES SAS,900413586,Atlántico,Barranquilla,2023-03-29
9,CO1.AWD.2315486,CO1.REQ.8427888,1606596857.00,732170105,CONSORCIO VIRREY ARC,901973576,None,Funza,2025-07-28


### Decisión sobre la clave natural

In [9]:
# T1.1 · Decisión sobre la clave natural (las cifras salen de las celdas anteriores)
def _fila(universo, clave):
    return diagnostico_claves[
        (diagnostico_claves["universo"] == universo) & (diagnostico_claves["clave candidata"] == clave)
    ].iloc[0]

clave_adj = _fila("solo adjudicados", "id_del_proceso + id_adjudicacion")
solo_awd = _fila("solo adjudicados", "id_adjudicacion")
exactos_adj = filas_adjudicadas - filas_adj_sin_exactos
sobrantes_conflicto = filas_adj_sin_exactos - int(clave_adj["claves distintas"])
cambian = cambios_adjudicacion[cambios_adjudicacion["adjudicaciones donde cambia"] > 0]
lista_cambian = ", ".join(f"`{c}` ({n:,})" for c, n in zip(cambian["columna"], cambian["adjudicaciones donde cambia"]))

display(Markdown(f"""
**Clave elegida: `id_del_proceso` + `id_adjudicacion`.** Un contrato adjudicado es una adjudicación (`CO1.AWD…`) dentro de un proceso (`CO1.REQ…`).

Por qué:
- `id_adjudicacion` solo tiene valor cuando `adjudicado = True` y nunca es nula en esas {filas_adjudicadas:,} filas.
- Hay **{int(clave_adj['claves distintas']):,}** parejas proceso + adjudicación distintas y **{int(solo_awd['claves distintas']):,}** adjudicaciones distintas: cada adjudicación pertenece a un único proceso.
- `id_del_proceso` solo no sirve: un mismo proceso puede tener varias adjudicaciones (lotes o varios contratistas), y deduplicar por proceso dejaría **{int(_fila('solo adjudicados', 'id_del_proceso')['claves distintas']):,}** filas, perdiendo adjudicaciones reales.
- `fecha_ultima_publicacion` no cambia dentro de un proceso, así que no sirve para elegir una "versión más reciente".

Por qué se repiten las {filas_adjudicadas - int(clave_adj['claves distintas']):,} filas sobrantes entre los adjudicados:
1. **Duplicados exactos:** {exactos_adj:,} filas idénticas en todas las columnas (por ejemplo, una misma adjudicación repetida 5 veces).
2. **Combinaciones de valor y proveedor:** {sobrantes_conflicto:,} filas en {total_adj_repetidas:,} adjudicaciones ({100 * total_adj_repetidas / int(clave_adj['claves distintas']):.2f} % del total) que repiten la adjudicación con otro valor u otro proveedor. Las columnas que cambian son: {lista_cambian}. El resto de columnas (entidad, proceso, modalidad, fechas de publicación) es idéntico. Las filas son el cruce de varios valores con varios proveedores (2 valores × 2 proveedores = 4 filas) y el dato abierto no indica qué valor corresponde a qué proveedor, así que **no se pueden sumar**: inflarían el valor contratado.

Regla para T1.2 (una fila por clave, con `row_number` y este orden):
1. `fecha_adjudicacion` no nula primero.
2. Mayor `valor_total_adjudicacion`.
3. Proveedor identificado primero: con `nit_proveedor_adjudicado` y luego con `codigo_proveedor`.
4. Desempate fijo: `codigo_proveedor`, `nit_proveedor_adjudicado` y `nombre_proveedor_adjudicado` en orden ascendente.

Las filas descartadas se guardan aparte con su motivo (`duplicado_exacto` o `version_alternativa`). Para los registros no adjudicados (`id_adjudicacion` nula) la clave queda en una fila por proceso.
"""))


**Clave elegida: `id_del_proceso` + `id_adjudicacion`.** Un contrato adjudicado es una adjudicación (`CO1.AWD…`) dentro de un proceso (`CO1.REQ…`).

Por qué:
- `id_adjudicacion` solo tiene valor cuando `adjudicado = True` y nunca es nula en esas 872,756 filas.
- Hay **547,418** parejas proceso + adjudicación distintas y **547,418** adjudicaciones distintas: cada adjudicación pertenece a un único proceso.
- `id_del_proceso` solo no sirve: un mismo proceso puede tener varias adjudicaciones (lotes o varios contratistas), y deduplicar por proceso dejaría **520,624** filas, perdiendo adjudicaciones reales.
- `fecha_ultima_publicacion` no cambia dentro de un proceso, así que no sirve para elegir una "versión más reciente".

Por qué se repiten las 325,338 filas sobrantes entre los adjudicados:
1. **Duplicados exactos:** 55,497 filas idénticas en todas las columnas (por ejemplo, una misma adjudicación repetida 5 veces).
2. **Combinaciones de valor y proveedor:** 269,841 filas en 16,616 adjudicaciones (3.04 % del total) que repiten la adjudicación con otro valor u otro proveedor. Las columnas que cambian son: `valor_total_adjudicacion` (16,110), `codigo_proveedor` (10,571), `nombre_proveedor_adjudicado` (10,571), `nit_proveedor_adjudicado` (7,893), `departamento_proveedor` (7,333), `ciudad_proveedor` (7,094), `fecha_adjudicacion` (163). El resto de columnas (entidad, proceso, modalidad, fechas de publicación) es idéntico. Las filas son el cruce de varios valores con varios proveedores (2 valores × 2 proveedores = 4 filas) y el dato abierto no indica qué valor corresponde a qué proveedor, así que **no se pueden sumar**: inflarían el valor contratado.

Regla para T1.2 (una fila por clave, con `row_number` y este orden):
1. `fecha_adjudicacion` no nula primero.
2. Mayor `valor_total_adjudicacion`.
3. Proveedor identificado primero: con `nit_proveedor_adjudicado` y luego con `codigo_proveedor`.
4. Desempate fijo: `codigo_proveedor`, `nit_proveedor_adjudicado` y `nombre_proveedor_adjudicado` en orden ascendente.

Las filas descartadas se guardan aparte con su motivo (`duplicado_exacto` o `version_alternativa`). Para los registros no adjudicados (`id_adjudicacion` nula) la clave queda en una fila por proceso.


## T1.2 Deduplicación determinista (capa silver)

Se deja una sola fila por clave (`id_del_proceso`, `id_adjudicacion`), con la regla aprobada en T1.1, usando `row_number` sobre una ventana con orden explícito. Las filas que no se conservan van a una tabla aparte con su motivo:
- `duplicado_exacto`: idéntica en todas las columnas a otra fila de la misma clave.
- `version_alternativa`: misma clave con otro valor, proveedor o fecha.

Salidas: `silver/secop/contratos` y `silver/secop/descartados`.

In [2]:
# T1.2 · Deduplicación determinista: una fila por (id_del_proceso, id_adjudicacion)
import pandas as pd

CLAVE_SILVER = ["id_del_proceso", "id_adjudicacion"]

# Orden de preferencia acordado en T1.1 (la primera fila de cada clave es la que se conserva)
ORDEN_SILVER = [
    F.col("fecha_adjudicacion").isNull().asc(),          # 1. con fecha de adjudicación primero
    F.col("valor_total_adjudicacion").desc_nulls_last(),  # 2. mayor valor adjudicado
    F.col("nit_proveedor_adjudicado").isNull().asc(),     # 3. proveedor con NIT...
    F.col("codigo_proveedor").isNull().asc(),             #    ...y luego con código
    F.col("codigo_proveedor").asc_nulls_last(),           # 4. desempate fijo
    F.col("nit_proveedor_adjudicado").asc_nulls_last(),
    F.col("nombre_proveedor_adjudicado").asc_nulls_last(),
]

def deduplicar(df):
    """Devuelve el DataFrame con 'rn' (1 = se conserva) y 'motivo_descarte'."""
    huella = F.xxhash64(*columnas_contenido)  # identifica filas idénticas en todas las columnas
    por_clave = Window.partitionBy(*CLAVE_SILVER).orderBy(*ORDEN_SILVER)
    por_clave_y_huella = Window.partitionBy(*CLAVE_SILVER, "_huella").orderBy(*ORDEN_SILVER)
    return (
        df.withColumn("_huella", huella)
        .withColumn("rn", F.row_number().over(por_clave))
        .withColumn("_rn_huella", F.row_number().over(por_clave_y_huella))
        .withColumn(
            "motivo_descarte",
            F.when(F.col("rn") == 1, F.lit(None).cast("string"))
             .when(F.col("_rn_huella") > 1, F.lit("duplicado_exacto"))
             .otherwise(F.lit("version_alternativa")),
        )
        .drop("_huella", "_rn_huella")
    )

def resumen(df):
    """Conteo, sumas de valores y suma de huellas: si dos ejecuciones dan lo mismo, el resultado es idéntico."""
    return df.agg(
        F.count("*").alias("filas"),
        F.sum("valor_total_adjudicacion").alias("suma_valor_adjudicado"),
        F.sum("precio_base").alias("suma_precio_base"),
        F.sum(F.xxhash64(*columnas_contenido).cast("decimal(38,0)")).alias("suma_huellas"),
    ).first().asDict()

### Escritura de silver

`RECONSTRUIR_SILVER = True` recalcula y escribe silver y los descartados; con `False` se reutiliza lo que ya está en MinIO.

In [3]:
# T1.2 · Escritura de silver y de las filas descartadas
# True la primera vez (escribe silver y descartados); False reutiliza lo que ya está en MinIO.
# Silver se escribió el 13-nov-2026 con True; la salida de abajo es la de esa ejecución.
RECONSTRUIR_SILVER = False

if RECONSTRUIR_SILVER:
    marcado = deduplicar(bronze).persist(StorageLevel.DISK_ONLY)
    (
        marcado.filter(F.col("rn") == 1)
        .drop("rn", "motivo_descarte")
        .write.mode(WRITE_MODE).option("compression", "snappy")
        .parquet(SILVER_PARQUET)
    )
    (
        marcado.filter(F.col("rn") > 1)
        .drop("rn")
        .write.mode(WRITE_MODE).option("compression", "snappy")
        .parquet(SILVER_DESCARTADOS_PARQUET)
    )
    marcado.unpersist()

silver = spark.read.parquet(SILVER_PARQUET)
descartados = spark.read.parquet(SILVER_DESCARTADOS_PARQUET)

### Criterios de aceptación

1. Silver tiene una fila por clave (se comprueba con `assert`).
2. Silver + descartadas = bronze: ninguna fila se pierde sin explicación.
3. Determinismo: se repite la deduplicación y se compara con lo escrito el conteo, la suma de `valor_total_adjudicacion`, la de `precio_base` y la suma de las huellas (`xxhash64`) de todas las filas. Si coinciden, las dos ejecuciones conservaron exactamente las mismas filas.

In [5]:
# T1.2 · Criterios de aceptación
filas_silver = silver.count()
filas_descartadas = descartados.count()
claves_distintas_bronze = bronze.select(*CLAVE_SILVER).distinct().count()

print(f"Bronze:                    {filas_bronze:,}")
print(f"Silver:                    {filas_silver:,}")
print(f"Descartadas:               {filas_descartadas:,}")
print(f"Claves distintas (bronze): {claves_distintas_bronze:,}")

# (1) Una fila por clave
assert filas_silver == claves_distintas_bronze, "Silver no tiene una fila por clave"
assert silver.groupBy(*CLAVE_SILVER).count().filter(F.col("count") > 1).count() == 0, "Hay claves repetidas en silver"
# Ninguna fila se pierde: silver + descartadas = bronze
assert filas_silver + filas_descartadas == filas_bronze, "Silver + descartadas no suma el bronze"

display(pd.DataFrame([r.asDict() for r in descartados.groupBy("motivo_descarte").count().orderBy("motivo_descarte").collect()]))

# (2) Determinismo: se repite la deduplicación en memoria y se compara con lo escrito
resumen_escrito = resumen(silver)
resumen_repetido = resumen(deduplicar(bronze).filter(F.col("rn") == 1).drop("rn", "motivo_descarte"))
display(pd.DataFrame([resumen_escrito, resumen_repetido], index=["silver escrito", "segunda ejecución"]))
assert resumen_escrito == resumen_repetido, "Dos ejecuciones dieron resultados distintos"
print("Criterios de aceptación de T1.2: OK")

Bronze:                    9,099,432
Silver:                    8,694,126
Descartadas:               405,306
Claves distintas (bronze): 8,694,126


,motivo_descarte,count
0,duplicado_exacto,135465
1,version_alternativa,269841


,filas,suma_valor_adjudicado,suma_precio_base,suma_huellas
silver escrito,8694126,570571474774174068.00,194476622903948512.00,-5729303209801678712062
segunda ejecución,8694126,570571474774174068.00,194476622903948512.00,-5729303209801678712062


Criterios de aceptación de T1.2: OK


## T1.3 Tabla gold de adjudicaciones

Gold parte de silver y conserva solo los contratos adjudicados (`adjudicado = True`) con fecha de adjudicación. Tiene una fila por adjudicación, con la entidad, el contratista, el valor y la fecha, más `anio_adjudicacion` y `anio_mes` para las agregaciones del Corte 3. Los identificadores siguen la regla del Punto 3 del Corte 2: la entidad por NIT y, si falta, por nombre; el contratista por NIT, luego por código y luego por nombre.

Un adjudicado no pasa a gold si le falta la fecha de adjudicación, si esa fecha es posterior a la descarga de los datos (2 de septiembre de 2026), si no se puede identificar la entidad o el contratista, o si el valor falta o es negativo. Esos registros se guardan en `gold/secop/excluidos` con su motivo.

Salida: `gold/secop/adjudicaciones`, en Parquet particionado por `anio_adjudicacion`.

In [4]:
# T1.3 · Construcción de gold: un registro por contrato adjudicado
FECHA_CORTE_DATOS = "2026-09-02"  # descarga del CSV de SECOP II; una adjudicación posterior es una fecha futura

def texto_normalizado(columna):
    """Mayúsculas, espacios simples y sin marcadores de ausencia (misma regla que el Punto 3 del Corte 2)."""
    valor = F.upper(F.trim(F.regexp_replace(F.col(columna).cast("string"), r"\s+", " ")))
    return F.when(~valor.isin("", "NO DEFINIDO", "NO DEFINIDA", "NO ADJUDICADO", "N/A", "NA", "NULL", "0", "-1"), valor)

# Identificadores: entidad por NIT y luego nombre; contratista por NIT, luego código y luego nombre (Punto 3 del Corte 2)
id_entidad = F.coalesce(
    F.concat(F.lit("NIT:"), texto_normalizado("nit_entidad")),
    F.concat(F.lit("NOMBRE:"), texto_normalizado("entidad")),
)
id_contratista = F.coalesce(
    F.concat(F.lit("NIT:"), texto_normalizado("nit_proveedor_adjudicado")),
    F.concat(F.lit("CODIGO:"), texto_normalizado("codigo_proveedor")),
    F.concat(F.lit("NOMBRE:"), texto_normalizado("nombre_proveedor_adjudicado")),
)

adjudicados_silver = silver.filter(F.col("adjudicado") == True)

candidatos_gold = adjudicados_silver.select(
    "id_del_proceso",
    "id_adjudicacion",
    id_entidad.alias("id_entidad"),
    F.col("nit_entidad"),
    F.col("entidad").alias("nombre_entidad"),
    "departamento_entidad",
    "ciudad_entidad",
    id_contratista.alias("id_contratista"),
    F.col("nombre_proveedor_adjudicado").alias("nombre_contratista"),
    F.col("departamento_proveedor").alias("departamento_contratista"),
    "modalidad_contratacion",
    "tipo_contrato",
    "valor_total_adjudicacion",
    F.to_date("fecha_adjudicacion").alias("fecha_adjudicacion"),
).withColumn(
    "motivo_exclusion",
    F.when(F.col("fecha_adjudicacion").isNull(), F.lit("sin_fecha_adjudicacion"))
     .when(F.col("fecha_adjudicacion") > F.lit(FECHA_CORTE_DATOS).cast("date"), F.lit("fecha_futura"))
     .when(F.col("id_entidad").isNull(), F.lit("sin_entidad"))
     .when(F.col("id_contratista").isNull(), F.lit("sin_contratista"))
     .when(F.col("valor_total_adjudicacion").isNull(), F.lit("sin_valor"))
     .when(F.col("valor_total_adjudicacion") < 0, F.lit("valor_negativo")),
)

gold_df = (
    candidatos_gold.filter(F.col("motivo_exclusion").isNull())
    .drop("motivo_exclusion")
    .withColumn("anio_adjudicacion", F.year("fecha_adjudicacion"))
    .withColumn("anio_mes", F.date_format("fecha_adjudicacion", "yyyy-MM"))
)
excluidos_gold = candidatos_gold.filter(F.col("motivo_exclusion").isNotNull())

### Escritura de gold

In [5]:
# True la primera vez (escribe gold y excluidos); False reutiliza lo que ya está en MinIO.
# Gold se escribió el 13-nov-2026 con True; la salida de abajo es la de esa ejecución.
RECONSTRUIR_GOLD = False

if RECONSTRUIR_GOLD:
    (
        gold_df
        .repartition("anio_adjudicacion")
        .write.mode(WRITE_MODE).option("compression", "snappy")
        .partitionBy("anio_adjudicacion")
        .parquet(GOLD_PARQUET)
    )
    excluidos_gold.write.mode(WRITE_MODE).option("compression", "snappy").parquet(GOLD_EXCLUIDOS_PARQUET)

gold = spark.read.parquet(GOLD_PARQUET)
excluidos = spark.read.parquet(GOLD_EXCLUIDOS_PARQUET)

### Esquema de gold

In [6]:
# T1.3 · Esquema documentado (tipo y significado de cada columna)
significado_gold = {
    "id_del_proceso": "Identificador del proceso de contratación en SECOP II (CO1.REQ…).",
    "id_adjudicacion": "Identificador de la adjudicación (CO1.AWD…). Con id_del_proceso forma la clave de gold.",
    "id_entidad": "Identificador de la entidad: 'NIT:<nit>' o, si falta, 'NOMBRE:<nombre normalizado>'.",
    "nit_entidad": "NIT de la entidad contratante, tal como viene del bronze.",
    "nombre_entidad": "Nombre de la entidad contratante.",
    "departamento_entidad": "Departamento de la entidad (filtro por región del dashboard).",
    "ciudad_entidad": "Ciudad de la entidad.",
    "id_contratista": "Identificador del contratista: 'NIT:', luego 'CODIGO:' y luego 'NOMBRE:' (regla del Punto 3 del Corte 2).",
    "nombre_contratista": "Nombre del proveedor adjudicado.",
    "departamento_contratista": "Departamento del proveedor adjudicado (puede ser nulo).",
    "modalidad_contratacion": "Modalidad de contratación del proceso.",
    "tipo_contrato": "Tipo de contrato.",
    "valor_total_adjudicacion": "Valor adjudicado en pesos colombianos (base de todos los montos del Corte 3).",
    "fecha_adjudicacion": "Fecha de adjudicación (sin hora).",
    "anio_mes": "Año y mes de adjudicación, 'AAAA-MM' (ventanas mensuales y gráficas).",
    "anio_adjudicacion": "Año de adjudicación; columna de partición de gold.",
}
esquema_gold = pd.DataFrame(
    [(f.name, f.dataType.simpleString(), significado_gold.get(f.name, "")) for f in gold.schema.fields],
    columns=["columna", "tipo", "significado"],
)
assert (esquema_gold["significado"] != "").all(), "Hay columnas sin documentar"
display(esquema_gold)

,columna,tipo,significado
0,id_del_proceso,string,Identificador del proceso de contratación en S...
1,id_adjudicacion,string,Identificador de la adjudicación (CO1.AWD…). C...
2,id_entidad,string,"Identificador de la entidad: 'NIT:<nit>' o, si..."
3,nit_entidad,bigint,"NIT de la entidad contratante, tal como viene ..."
4,nombre_entidad,string,Nombre de la entidad contratante.
5,departamento_entidad,string,Departamento de la entidad (filtro por región ...
6,ciudad_entidad,string,Ciudad de la entidad.
7,id_contratista,string,"Identificador del contratista: 'NIT:', luego '..."
8,nombre_contratista,string,Nombre del proveedor adjudicado.
9,departamento_contratista,string,Departamento del proveedor adjudicado (puede s...


### Criterios de aceptación

1. El esquema está documentado con el tipo y el significado de cada columna (celda anterior).
2. No hay valores adjudicados negativos ni fechas anteriores a 1900 o posteriores al corte de los datos (se comprueba con `assert`).
3. Los registros excluidos están contados por motivo, y gold + excluidos = adjudicados de silver.

In [7]:
# T1.3 · Criterios de aceptación
filas_gold = gold.count()
filas_excluidas = excluidos.count()
filas_adjudicadas_silver = adjudicados_silver.count()

print(f"Adjudicados en silver:  {filas_adjudicadas_silver:,}")
print(f"Gold:                   {filas_gold:,}")
print(f"Excluidos:              {filas_excluidas:,}")
assert filas_gold + filas_excluidas == filas_adjudicadas_silver, "Gold + excluidos no suma los adjudicados de silver"
assert gold.select("id_del_proceso", "id_adjudicacion").distinct().count() == filas_gold, "Clave repetida en gold"

# (3) Registros excluidos, por motivo
display(pd.DataFrame([r.asDict() for r in excluidos.groupBy("motivo_exclusion").count().orderBy(F.desc("count")).collect()]))

# (2) Valores y fechas fuera de rango
rangos = gold.agg(
    F.min("valor_total_adjudicacion").alias("valor_minimo"),
    F.max("valor_total_adjudicacion").alias("valor_maximo"),
    F.sum(F.when(F.col("valor_total_adjudicacion") == 0, 1).otherwise(0)).alias("valores_en_cero"),
    F.min("fecha_adjudicacion").alias("fecha_minima"),
    F.max("fecha_adjudicacion").alias("fecha_maxima"),
).first()
print(rangos)
assert rangos["valor_minimo"] >= 0, "Hay valores adjudicados negativos"
assert str(rangos["fecha_minima"]) >= "1900-01-01", "Hay fechas anteriores a 1900"
assert str(rangos["fecha_maxima"]) <= FECHA_CORTE_DATOS, "Hay fechas posteriores al corte de los datos"

# Contratos por año, para ver la cobertura temporal de gold
display(pd.DataFrame([r.asDict() for r in gold.groupBy("anio_adjudicacion")
                      .agg(F.count("*").alias("contratos"), F.sum("valor_total_adjudicacion").alias("valor_total"))
                      .orderBy("anio_adjudicacion").collect()]))
print("Criterios de aceptación de T1.3: OK")

Adjudicados en silver:  547,418
Gold:                   530,192
Excluidos:              17,226


,motivo_exclusion,count
0,sin_fecha_adjudicacion,17053
1,sin_contratista,173


Row(valor_minimo=Decimal('0.00'), valor_maximo=Decimal('89069800047730398.00'), valores_en_cero=1527, fecha_minima=datetime.date(2015, 6, 11), fecha_maxima=datetime.date(2026, 9, 1))


,anio_adjudicacion,contratos,valor_total
0,2015,29,9144206616.00
1,2016,1859,70758269283056.00
2,2017,18284,67600174083503.00
3,2018,32141,13927437992466482.00
4,2019,33193,8857511584117998.00
5,2020,39160,6999931462533473.00
6,2021,56100,13540647558190157.00
7,2022,66717,100469226243036004.00
8,2023,71544,8869840410436041.00
9,2024,74067,6620289880191104.00


Criterios de aceptación de T1.3: OK


In [8]:
# T1.3 · Observaciones para T1.5: valores atípicos en gold (no se modifican aquí)
mayores = gold.orderBy(F.desc("valor_total_adjudicacion")).select(
    "id_adjudicacion", "nombre_entidad", "nombre_contratista", "fecha_adjudicacion", "valor_total_adjudicacion"
).limit(10).collect()
display(pd.DataFrame([r.asDict() for r in mayores]))

umbral_billon = 10**12  # un billón de pesos
sobre_umbral = gold.filter(F.col("valor_total_adjudicacion") > umbral_billon).agg(
    F.count("*").alias("n"), F.sum("valor_total_adjudicacion").alias("suma")).first()
total_gold = gold.agg(F.sum("valor_total_adjudicacion")).first()[0]
display(Markdown(f"""
**Observaciones para T1.5 (valores atípicos).** Gold conserva los valores tal como vienen de SECOP II; la decisión de marcarlos o excluirlos es de T1.5.
- {rangos['valores_en_cero']:,} adjudicaciones tienen valor 0.
- {sobre_umbral['n']:,} adjudicaciones superan un billón de pesos y suman el {100 * sobre_umbral['suma'] / total_gold:.2f} % del valor total de gold. El máximo es {rangos['valor_maximo']:,.0f} pesos, varias veces el presupuesto anual de la Nación, así que se trata de errores de captura.
- 2026 solo llega hasta {rangos['fecha_maxima']} (fecha de descarga de los datos), así que es un año incompleto.
"""))

,id_adjudicacion,nombre_entidad,nombre_contratista,fecha_adjudicacion,valor_total_adjudicacion
0,CO1.AWD.1445345,EAG,URBE,2022-10-18,89069800047730398.00
1,CO1.AWD.1192019,INSTITUTO MUNICIPAL DEL DEPORTE Y LA RECREACIO...,MAQUINARIAS PARA AREAS VERDES SA MAQUIVER SA,2021-12-09,12858450316000000.00
2,CO1.AWD.348301,MUNICIPIO DE FIRAVITOBA,FUNDACION SOÑANDO POR NUESTRO MUNDO,2018-08-29,11877456456257536.00
3,CO1.AWD.1408949,ALCALDIA MUNICIPAL DE BUGA,RODRIGO VALENCIA VILLARREAL,2022-09-08,11015179929862500.00
4,CO1.AWD.570328,MUNICIPIO DE ENVIGADO*,CORPORACIÓN EVG EXTEMO,2019-08-21,8839189460374700.00
5,CO1.AWD.1769384,DEPARTAMENTO DE SANTANDER,CARLOS AUGUSTO HERNANDEZ SANCHEZ,2023-11-15,8811763956840000.00
6,CO1.AWD.2679736,ESE HOSPITAL LOCAL SAN JOSE DE PUEBLO VIEJO,G&C MEDICALS S.A.S,2026-08-31,8054481856630300.00
7,CO1.AWD.2253730,IMDERCOR,FUNDACIÓN PARA LA INNOVACIÓN DE LAS CIENCIAS D...,2025-05-22,6514265521000000.00
8,CO1.AWD.1840933,INSTITUCION EDUCATIVA ELISEO PAYAN,CIFRAS ASESORÍAS Y PROYECTOS CIASPRO S.A.S.,2024-02-19,6453840000000000.00
9,CO1.AWD.751206,DEPARTAMENTO DE CUNDINAMARCA- SECRETARIA DE SALUD,LABCARE DE COLOMBIA SAS,2020-05-20,6158661742599964.00



**Observaciones para T1.5 (valores atípicos).** Gold conserva los valores tal como vienen de SECOP II; la decisión de marcarlos o excluirlos es de T1.5.
- 1,527 adjudicaciones tienen valor 0.
- 48 adjudicaciones superan un billón de pesos y suman el 99.79 % del valor total de gold. El máximo es 89,069,800,047,730,398 pesos, varias veces el presupuesto anual de la Nación, así que se trata de errores de captura.
- 2026 solo llega hasta 2026-09-01 (fecha de descarga de los datos), así que es un año incompleto.
